# DeepSeek Agent 实战：小红书爆款文案生成助手

本 Notebook 将指导您如何使用 DeepSeek LLM 构建一个能够生成小红书爆款文案的智能 Agent。我们将从需求拆解开始，逐步定义 Agent 的系统提示词 (System Prompt)、外部工具 (Tools)，并实现其核心的工作流程，最终生成符合小红书平台特点的文案。

## 1. 环境准备与DeepSeek API配置

In [13]:
import os
from openai import OpenAI

# 建议将 API Key 设置为环境变量，避免直接暴露在代码中
# 从环境变量获取 DeepSeek API Key
api_key = os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    raise ValueError("请设置 DEEPSEEK_API_KEY 环境变量")

# 初始化 DeepSeek 客户端

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",  # DeepSeek API 的基地址
)

## 2. 需求拆解与Agent任务规划

#### 用户痛点与核心需求：
*   **效率低下：** 人工创作周期长，难以满足高频发布需求。
*   **创意瓶颈：** 难以持续产出新颖、吸引人的爆款创意。
*   **趋势捕捉难：** 实时流行元素难以快速融入文案。
*   **平台特性把握：** 小红书特有风格（标题、正文、标签、表情）难以精准复制。

#### “爆款”文案的特征：
*   **强吸引力标题：** 制造好奇、痛点共鸣、利益点清晰。
*   **沉浸式正文：** 真实体验分享、细节描述、情感共鸣。
*   **精准且多样标签：** 热门话题、品牌词、产品词、垂直领域词。
*   **生动表情符号：** 增强表达力，提升活泼感。
*   **清晰的行动召唤 (CTA)。**

#### Agent 任务规划：核心工作流
1.  **用户指令接收：** 接收产品信息、主题、风格等。
2.  **信息收集 (Web Search/DB Query)：** 实时搜索行业趋势、热门话题、竞品分析、产品卖点。
3.  **内容构思与初稿生成 (LLM)：** 结合所有信息，撰写标题、正文、标签、表情符号。
4.  **风格与格式优化 (LLM)：** 根据小红书平台特点和指定风格，对文案进行润色和结构调整。
5.  **最终输出：** 呈现完整文案。

In [14]:
# 安装必要的依赖包
!pip install ddgs requests beautifulsoup4


## 3. 爆款文案生成逻辑与 Prompt 设计

### 3.1 System Prompt (系统提示词)

System Prompt 是 Agent 的“大脑”和“行为准则”。它定义了 Agent 的角色、目标以及工作方式。我们将采用 `Thought-Action-Observation` (ReAct) 模式来引导 DeepSeek 的推理过程。

In [15]:
SYSTEM_PROMPT = """
你是一个资深的小红书爆款文案专家，擅长结合最新潮流和产品卖点，创作引人入胜、高互动、高转化的笔记文案。

你的任务是根据用户提供的产品和需求，生成包含标题、正文、相关标签和表情符号的完整小红书笔记。

请始终采用'Thought-Action-Observation'模式进行推理和行动。文案风格需活泼、真诚、富有感染力。当完成任务后，请以JSON格式直接输出最终文案，格式如下：
```json
{
  "title": "小红书标题",
  "body": "小红书正文",
  "hashtags": ["#标签1", "#标签2", "#标签3", "#标签4", "#标签5"],
  "emojis": ["✨", "🔥", "💖"]
}
```
在生成文案前，请务必先思考并收集足够的信息。
"""

### 3.2 Tools (工具定义)

Agent 的“双手”由一系列可调用的工具组成。这些工具扩展了 LLM 的能力，使其能够获取实时信息、查询数据库或执行特定操作。在这里，我们定义了三个核心工具：

*   `search_web`: 用于搜索互联网上的实时信息，如最新趋势、用户评价等。
*   `query_product_database`: 用于查询产品数据库，获取产品的详细卖点和特点。**此工具为模拟**。
*   `generate_emoji`: 用于根据文案内容生成恰当的表情符号。**此工具为模拟**。

In [16]:
TOOLS_DEFINITION = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "搜索互联网上的实时信息，用于获取最新新闻、流行趋势、用户评价、行业报告等。请确保搜索关键词精确，避免宽泛的查询。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "要搜索的关键词或问题，例如'最新小红书美妆趋势'或'深海蓝藻保湿面膜 用户评价'"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "query_product_database",
            "description": "查询内部产品数据库，获取指定产品的详细卖点、成分、适用人群、使用方法等信息。",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "要查询的产品名称，例如'深海蓝藻保湿面膜'"
                    }
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_xiaohongshu_trends",
            "description": "专门搜索小红书平台相关的热门趋势、爆款内容和用户反馈，获取更精准的小红书风格参考。",
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "要搜索的关键词，例如'美妆'、'护肤'、'面膜'等"
                    }
                },
                "required": ["keyword"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_emoji",
            "description": "根据提供的文本内容，生成一组适合小红书风格的表情符号。",
            "parameters": {
                "type": "object",
                "properties": {
                    "context": {
                        "type": "string",
                        "description": "文案的关键内容或情感，例如'惊喜效果'、'补水保湿'"
                    }
                },
                "required": ["context"]
            }
        }
    }
]

### 3.3 工具实现

现在我们实现Agent的核心工具：

- **真实网络搜索**: 使用DuckDuckGo提供真实的网络搜索功能，获取最新趋势和信息
- **小红书趋势搜索**: 专门针对小红书平台的精准搜索，获取更相关的参考内容  
- **产品数据库查询**: 模拟内部产品数据库（在实际应用中可替换为真实数据库API）
- **表情符号生成**: 根据内容生成适合的表情符号

In [17]:
import random # 用于模拟生成表情
import time # 用于模拟网络延迟
# 使用正确的包名
try:
    from ddgs import DDGS
except ImportError:
    # 备用方案：使用旧包名（如果存在）
    try:
        from duckduckgo_search import DDGS
    except ImportError:
        print("⚠️ 需要安装搜索包: pip install ddgs")
        DDGS = None
import requests
from bs4 import BeautifulSoup
import re

def real_search_web(query: str, max_results: int = 5) -> str:
    """真实的网页搜索工具，使用DuckDuckGo进行搜索，包含备用方案。"""
    print(f"[Tool Call] 搜索网页：{query}")
    
    # 检查DDGS是否可用
    if DDGS is None:
        return fallback_search_response(query)
    
    # 尝试多种搜索策略
    search_attempts = [
        lambda: _ddgs_search_text(query, max_results),
        lambda: _ddgs_search_instant(query),
        lambda: fallback_search_response(query)
    ]
    
    for attempt_num, search_func in enumerate(search_attempts, 1):
        try:
            print(f"   尝试搜索方法 {attempt_num}...")
            result = search_func()
            if result and "搜索失败" not in result:
                return result
        except Exception as e:
            print(f"   搜索方法 {attempt_num} 失败: {e}")
            continue
    
    # 所有方法都失败，返回备用响应
    return fallback_search_response(query)

def _ddgs_search_text(query: str, max_results: int = 5) -> str:
    """使用DDGS文本搜索"""
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=max_results, backend="api"))
        
        if not results:
            return None
        
        # 整理搜索结果
        formatted_results = []
        for i, result in enumerate(results, 1):
            title = result.get('title', '无标题')
            body = result.get('body', '无描述')
            url = result.get('href', '')
            
            # 清理文本，移除多余的空白字符
            title = re.sub(r'\s+', ' ', title).strip()
            body = re.sub(r'\s+', ' ', body).strip()
            
            formatted_results.append(f"{i}. {title}\n   {body}\n   来源: {url}")
        
        search_summary = f"搜索关键词: {query}\n找到 {len(results)} 条相关结果:\n\n" + "\n\n".join(formatted_results)
        
        # 限制返回内容长度，避免token过多
        if len(search_summary) > 2000:
            search_summary = search_summary[:2000] + "...\n[结果已截断]"
            
        return search_summary

def _ddgs_search_instant(query: str) -> str:
    """使用DDGS即时搜索作为备用"""
    with DDGS() as ddgs:
        results = ddgs.answers(query)
        if results:
            return f"搜索关键词: {query}\n即时回答: {results[0].get('text', '无结果')}"
    return None

def fallback_search_response(query: str) -> str:
    """当搜索功能不可用时的备用响应"""
    print(f"   使用备用知识库响应...")
    
    # 基于查询关键词提供相关信息
    if "cursor" in query.lower():
        return """搜索关键词: cursor 相关信息
        
基于知识库的信息:
1. Cursor是一款AI驱动的代码编辑器
   功能特点：智能补全、多语言支持、团队协作
   
2. 主要优势：提升编程效率、AI辅助编程、现代化界面
   适用人群：程序员、开发者、学生
   
3. 相关趋势：AI编程工具越来越受欢迎，提升开发效率是关键需求
   
[备用知识库响应]"""
    
    elif any(keyword in query.lower() for keyword in ["美妆", "护肤", "面膜", "精华"]):
        return f"""搜索关键词: {query}
        
基于知识库的美妆护肤信息:
1. 当前流行趋势：早C晚A护肤理念、成分党兴起、敏感肌友好产品
2. 用户关注点：产品成分、使用效果、性价比
3. 小红书热门话题：种草推荐、使用心得、对比测评

[备用知识库响应]"""
    
    else:
        return f"""搜索关键词: {query}
        
由于网络搜索暂时不可用，将基于产品名称和常见市场趋势生成文案。
建议关注：
1. 产品核心功能和亮点
2. 目标用户群体需求
3. 当前市场流行趋势

[备用知识库响应]"""

def search_xiaohongshu_trends(keyword: str) -> str:
    """专门搜索小红书相关趋势的增强版搜索函数，包含备用方案。"""
    print(f"[Tool Call] 搜索小红书趋势：{keyword}")
    
    # 检查DDGS是否可用
    if DDGS is None:
        return fallback_xiaohongshu_response(keyword)
    
    # 构造更精准的搜索查询
    search_queries = [
        f"小红书 {keyword} 热门 趋势",
        f"{keyword} 小红书 爆款 推荐",
        f"小红书 {keyword} 种草 攻略"
    ]
    
    all_results = []
    
    try:
        print(f"   正在搜索小红书{keyword}趋势...")
        with DDGS() as ddgs:
            for query in search_queries:
                try:
                    results = list(ddgs.text(query, max_results=3, backend="api"))
                    all_results.extend(results)
                    # 添加短暂延迟，避免请求过于频繁
                    time.sleep(0.5)
                except Exception as e:
                    print(f"   查询'{query}'失败: {e}")
                    continue
        
        if not all_results:
            print(f"   网络搜索无结果，使用备用方案...")
            return fallback_xiaohongshu_response(keyword)
        
        # 去重并整理结果
        seen_titles = set()
        unique_results = []
        for result in all_results:
            title = result.get('title', '')
            if title not in seen_titles:
                seen_titles.add(title)
                unique_results.append(result)
        
        # 格式化结果
        formatted_results = []
        for i, result in enumerate(unique_results[:8], 1):  # 最多8个结果
            title = result.get('title', '无标题')
            body = result.get('body', '无描述')
            
            # 清理和截断文本
            title = re.sub(r'\s+', ' ', title).strip()
            body = re.sub(r'\s+', ' ', body).strip()
            if len(body) > 150:
                body = body[:150] + "..."
                
            formatted_results.append(f"{i}. {title}\n   {body}")
        
        summary = f"小红书'{keyword}'相关趋势分析:\n\n" + "\n\n".join(formatted_results)
        
        # 控制总长度
        if len(summary) > 2500:
            summary = summary[:2500] + "...\n[内容已截断]"
            
        return summary
        
    except Exception as e:
        print(f"搜索小红书趋势时发生错误: {e}")
        return fallback_xiaohongshu_response(keyword)

def fallback_xiaohongshu_response(keyword: str) -> str:
    """小红书趋势搜索的备用响应"""
    print(f"   使用小红书备用知识库...")
    
    # 根据关键词提供相关的小红书趋势信息
    if keyword.lower() in ["cursor", "代码", "编程", "开发"]:
        return f"""小红书'{keyword}'相关趋势分析:

1. 程序员日常分享
   内容：编程工具推荐、代码技巧分享、开发环境配置

2. 技术好物种草
   热门话题：提升效率的工具、AI辅助编程、职场技能

3. 学习成长记录
   流行内容：编程学习路径、项目经验分享、技术书籍推荐

[基于小红书备用知识库]"""

    elif any(word in keyword.lower() for word in ["美妆", "护肤", "面膜", "精华", "化妆"]):
        return f"""小红书'{keyword}'相关趋势分析:

1. 成分党崛起
   热门内容：成分分析、功效对比、敏感肌友好产品

2. 真实使用体验
   流行形式：空瓶记、使用心得、前后对比图

3. 平价替代推荐
   热门话题：大牌平替、性价比好物、学生党必买

4. 季节性护肤
   内容重点：换季护肤、干皮救星、油皮控油

[基于小红书备用知识库]"""

    else:
        return f"""小红书'{keyword}'相关趋势分析:

1. 真实体验分享
   小红书用户更偏爱真实的使用体验和详细的产品测评

2. 种草与拔草
   流行的内容形式包括好物推荐、踩雷警告、对比测评

3. 生活方式展示
   用户喜欢看到产品如何融入日常生活场景

4. 互动性内容
   问答形式、投票选择、经验分享等高互动内容更受欢迎

[基于小红书备用知识库]"""

def mock_search_web(query: str) -> str:
    """模拟网页搜索工具，返回预设的搜索结果。"""
    print(f"[Tool Call] 模拟搜索网页：{query}")
    time.sleep(1) # 模拟网络延迟
    if "小红书美妆趋势" in query:
        return "近期小红书美妆流行'多巴胺穿搭'、'早C晚A'护肤理念、'伪素颜'妆容，热门关键词有#氛围感、#抗老、#屏障修复。"
    elif "保湿面膜" in query:
        return "小红书保湿面膜热门话题：沙漠干皮救星、熬夜急救面膜、水光肌养成。用户痛点：卡粉、泛红、紧绷感。"
    elif "深海蓝藻保湿面膜" in query:
        return "关于深海蓝藻保湿面膜的用户评价：普遍反馈补水效果好，吸收快，对敏感肌友好。有用户提到价格略高，但效果值得。"
    else:
        return f"未找到关于 '{query}' 的特定信息，但市场反馈通常关注产品成分、功效和用户体验。"

def mock_query_product_database(product_name: str) -> str:
    """模拟查询产品数据库，返回预设的产品信息。"""
    print(f"[Tool Call] 模拟查询产品数据库：{product_name}")
    time.sleep(0.5) # 模拟数据库查询延迟
    if "深海蓝藻保湿面膜" in product_name:
        return "深海蓝藻保湿面膜：核心成分为深海蓝藻提取物，富含多糖和氨基酸，能深层补水、修护肌肤屏障、舒缓敏感泛红。质地清爽不粘腻，适合所有肤质，尤其适合干燥、敏感肌。规格：25ml*5片。"
    elif "美白精华" in product_name:
        return "美白精华：核心成分是烟酰胺和VC衍生物，主要功效是提亮肤色、淡化痘印、改善暗沉。质地轻薄易吸收，适合需要均匀肤色的人群。"
    else:
        return f"产品数据库中未找到关于 '{product_name}' 的详细信息。"

def mock_generate_emoji(context: str) -> list:
    """模拟生成表情符号，根据上下文提供常用表情。"""
    print(f"[Tool Call] 模拟生成表情符号，上下文：{context}")
    time.sleep(0.2) # 模拟生成延迟
    if "补水" in context or "水润" in context or "保湿" in context:
        return ["💦", "💧", "🌊", "✨"]
    elif "惊喜" in context or "哇塞" in context or "爱了" in context:
        return ["💖", "😍", "🤩", "💯"]
    elif "熬夜" in context or "疲惫" in context:
        return ["😭", "😮‍💨", "😴", "💡"]
    elif "好物" in context or "推荐" in context:
        return ["✅", "👍", "⭐", "🛍️"]
    else:
        return random.sample(["✨", "🔥", "💖", "💯", "🎉", "👍", "🤩", "💧", "🌿"], k=min(5, len(context.split())))

# 将工具函数映射到一个字典，方便通过名称调用
available_tools = {
    "search_web": real_search_web,
    "search_xiaohongshu_trends": search_xiaohongshu_trends,
    "query_product_database": mock_query_product_database,
    "generate_emoji": mock_generate_emoji,
}

In [12]:
### 🔧 搜索功能修复与改进

**问题诊断:**
1. **包名问题**: `duckduckgo_search` 已重命名为 `ddgs`
2. **网络连接问题**: DuckDuckGo API可能临时不可用
3. **缺乏备用方案**: 搜索失败时没有降级处理

**修复方案:**
1. **智能包导入**: 优先使用新包名`ddgs`，自动降级到旧包名
2. **多重搜索策略**: 文本搜索 → 即时搜索 → 备用知识库
3. **智能备用响应**: 基于关键词提供相关的行业知识
4. **详细错误日志**: 更好的调试信息和状态提示

**优势:**
- ✅ **高可用性**: 即使网络搜索失败也能正常工作
- ✅ **智能降级**: 自动选择最佳可用方案
- ✅ **内容质量**: 备用知识库提供有价值的行业信息
- ✅ **用户体验**: 透明的状态反馈，用户了解当前使用的数据源


SyntaxError: invalid character '，' (U+FF0C) (1175638038.py, line 9)

In [ ]:
# 🔧 修复搜索功能并测试
print("=== 🔧 测试修复后的搜索功能 ===")

# 测试普通网络搜索
print("\n1. 测试修复后的网络搜索:")
try:
    test_result = real_search_web("cursor 代码编辑器", max_results=2)
    print("✅ 搜索成功!")
    print(test_result[:300] + "..." if len(test_result) > 300 else test_result)
except Exception as e:
    print(f"❌ 搜索测试失败: {e}")

print("\n" + "="*50)

# 测试小红书专门搜索
print("\n2. 测试修复后的小红书趋势搜索:")
try:
    xiaohongshu_result = search_xiaohongshu_trends("编程")
    print("✅ 小红书趋势搜索成功!")
    print(xiaohongshu_result[:300] + "..." if len(xiaohongshu_result) > 300 else xiaohongshu_result)
except Exception as e:
    print(f"❌ 小红书趋势搜索测试失败: {e}")

print("\n" + "="*50)
print("🎉 搜索功能修复完成！现在包含了智能备用方案，即使网络搜索失败也能正常工作。")


In [ ]:
### 3.4 真实搜索功能测试

现在我们已经集成了真实的网络搜索功能！让我们测试一下：

- **DuckDuckGo搜索**: 免费、无需API密钥的真实网络搜索
- **智能结果整理**: 自动清理和格式化搜索结果
- **小红书专门搜索**: 针对小红书平台优化的搜索策略

**重要优势:**
- ✅ 获取实时、真实的网络信息
- ✅ 捕捉最新的流行趋势和用户反馈  
- ✅ 生成更准确、更有吸引力的文案内容
- ✅ 免费使用，无API限制


In [ ]:
# 测试真实搜索功能
print("=== 测试真实网络搜索功能 ===")

# 测试普通网络搜索
print("\n1. 测试普通网络搜索:")
test_result = real_search_web("小红书美妆趋势2024", max_results=3)
print(test_result)

print("\n" + "="*50)

# 测试小红书专门搜索
print("\n2. 测试小红书趋势搜索:")
xiaohongshu_result = search_xiaohongshu_trends("面膜")
print(xiaohongshu_result)


In [ ]:
## 🎉 升级完成：从Mock到真实搜索

**主要改进:**

1. **集成DuckDuckGo真实搜索** - 替换了原来的模拟搜索功能
2. **添加专门的小红书趋势搜索** - 更精准地获取小红书相关内容
3. **智能搜索结果处理** - 自动清理、格式化和去重搜索结果
4. **错误处理和容错机制** - 确保搜索失败时的优雅降级

**现在Agent将能够:**
- 🔍 获取真实的市场趋势和用户反馈
- 📈 捕捉最新的小红书流行话题
- 💡 基于真实数据生成更有说服力的文案
- 🎯 提供更准确的产品定位和营销角度

让我们继续使用升级后的Agent来生成小红书文案吧！


## 4. 实战：构建小红书文案生成 Agent

现在，我们将把 System Prompt、工具定义和模拟工具函数整合起来，构建出能够自动执行的 DeepSeek Agent 工作流。核心是 `generate_rednote` 函数，它通过一个循环来模拟 Agent 的 `Thought-Action-Observation` 过程。

In [18]:
import json
import re

def generate_rednote(product_name: str, tone_style: str = "活泼甜美", max_iterations: int = 5) -> str:
    """
    使用 DeepSeek Agent 生成小红书爆款文案。
    
    Args:
        product_name (str): 要生成文案的产品名称。
        tone_style (str): 文案的语气和风格，如"活泼甜美"、"知性"、"搞怪"等。
        max_iterations (int): Agent 最大迭代次数，防止无限循环。
        
    Returns:
        str: 生成的爆款文案（JSON 格式字符串）。
    """
    
    print(f"\n🚀 启动小红书文案生成助手，产品：{product_name}，风格：{tone_style}\n")
    
    # 存储对话历史，包括系统提示词和用户请求
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"请为产品「{product_name}」生成一篇小红书爆款文案。要求：语气{tone_style}，包含标题、正文、至少5个相关标签和5个表情符号。请以完整的JSON格式输出，并确保JSON内容用markdown代码块包裹（例如：```json{{...}}```）。"}
    ]
    
    iteration_count = 0
    final_response = None
    
    while iteration_count < max_iterations:
        iteration_count += 1
        print(f"-- Iteration {iteration_count} --")
        
        try:
            # 调用 DeepSeek API，传入对话历史和工具定义
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=messages,
                tools=TOOLS_DEFINITION, # 告知模型可用的工具
                tool_choice="auto" # 允许模型自动决定是否使用工具
            )

            response_message = response.choices[0].message
            
            # **ReAct模式：处理工具调用**
            if response_message.tool_calls: # 如果模型决定调用工具
                print("Agent: 决定调用工具...")
                messages.append(response_message) # 将工具调用信息添加到对话历史
                
                tool_outputs = []
                for tool_call in response_message.tool_calls:
                    function_name = tool_call.function.name
                    # 确保参数是合法的JSON字符串，即使工具不要求参数，也需要传递空字典
                    function_args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}

                    print(f"Agent Action: 调用工具 '{function_name}'，参数：{function_args}")
                    
                    # 查找并执行对应的模拟工具函数
                    if function_name in available_tools:
                        tool_function = available_tools[function_name]
                        tool_result = tool_function(**function_args)
                        print(f"Observation: 工具返回结果：{tool_result}")
                        tool_outputs.append({
                            "tool_call_id": tool_call.id,
                            "role": "tool",
                            "content": str(tool_result) # 工具结果作为字符串返回
                        })
                    else:
                        error_message = f"错误：未知的工具 '{function_name}'"
                        print(error_message)
                        tool_outputs.append({
                            "tool_call_id": tool_call.id,
                            "role": "tool",
                            "content": error_message
                        })
                messages.extend(tool_outputs) # 将工具执行结果作为 Observation 添加到对话历史
                
            # **ReAct 模式：处理最终内容**
            elif response_message.content: # 如果模型直接返回内容（通常是最终答案）
                print(f"[模型生成结果] {response_message.content}")
                
                # --- START: 添加 JSON 提取和解析逻辑 ---
                json_string_match = re.search(r"```json\s*(\{.*\})\s*```", response_message.content, re.DOTALL)
                
                if json_string_match:
                    extracted_json_content = json_string_match.group(1)
                    try:
                        final_response = json.loads(extracted_json_content)
                        print("Agent: 任务完成，成功解析最终JSON文案。")
                        return json.dumps(final_response, ensure_ascii=False, indent=2)
                    except json.JSONDecodeError as e:
                        print(f"Agent: 提取到JSON块但解析失败: {e}")
                        print(f"尝试解析的字符串:\n{extracted_json_content}")
                        messages.append(response_message) # 解析失败，继续对话
                else:
                    # 如果没有匹配到 ```json 块，尝试直接解析整个 content
                    try:
                        final_response = json.loads(response_message.content)
                        print("Agent: 任务完成，直接解析最终JSON文案。")
                        return json.dumps(final_response, ensure_ascii=False, indent=2)
                    except json.JSONDecodeError:
                        print("Agent: 生成了非JSON格式内容或非Markdown JSON块，可能还在思考或出错。")
                        messages.append(response_message) # 非JSON格式，继续对话
                # --- END: 添加 JSON 提取和解析逻辑 ---
            else:
                print("Agent: 未知响应，可能需要更多交互。")
                break
                
        except Exception as e:
            print(f"调用 DeepSeek API 时发生错误: {e}")
            break
    
    print("\n⚠️ Agent 达到最大迭代次数或未能生成最终文案。请检查Prompt或增加迭代次数。")
    return "未能成功生成文案。"

## 5. 实际测试与文案生成

现在，让我们调用我们构建的 `generate_rednote` 函数，看看它能生成什么样的爆款文案！

In [19]:
# 测试案例 1: 深海蓝藻保湿面膜
product_name_1 = "cursor"
tone_style_1 = "专业严谨"
result_1 = generate_rednote(product_name_1, tone_style_1)

print("\n--- 生成的文案 1 ---")
print(result_1)


🚀 启动小红书文案生成助手，产品：cursor，风格：专业严谨

-- Iteration 1 --
Agent: 决定调用工具...
Agent Action: 调用工具 'query_product_database'，参数：{'product_name': 'cursor'}
[Tool Call] 模拟查询产品数据库：cursor
Observation: 工具返回结果：产品数据库中未找到关于 'cursor' 的详细信息。
-- Iteration 2 --
Agent: 决定调用工具...
Agent Action: 调用工具 'search_web'，参数：{'query': 'cursor 产品介绍 小红书'}
[Tool Call] 搜索网页：cursor 产品介绍 小红书
   尝试搜索方法 1...
Observation: 工具返回结果：搜索关键词: cursor 产品介绍 小红书
找到 5 条相关结果:

1. Cursor +Deepseek，搞定 小 红 书 文案的「配图」！ - 53AI-AI...
   Cursor +Deepseek，搞定 小 红 书 文案的「配图」！打开 cursor 的chat（command+i）.复制之前的 小 红 书 文案，然后在整个文案前面加一句话“根据下面的文案，生成一个 小 红 书 风格的图片，html格式，注意尺寸，字要少，视觉效果.
   来源: https://www.53ai.com/news/neirongchuangzuo/2025021350164.html

2. 行为分析-PostHog...
   WordPress & WooCommerce 介 绍 .posthog埋点 介 绍 . 用户活跃情况分析.
   来源: https://liangdabiao.com/docs/行为分析-posthog应用

3. 《糖心Vlog：魅惑妲己沦为单男胯下淫物-媛媛酱》详情 介 绍 -糖心Vlog...
   国 产 推荐...
   来源: https://hd.huaduzy.org/voddetail/14362.html

4. nvhai27.top/index.php/vod/detail/id/72908.html
   台北12岁婀娜多姿的 小 女孩自拍展示并 介 

In [13]:
# 测试案例 2: 美白精华
product_name_2 = "美白精华"
tone_style_2 = "知性温柔"
result_2 = generate_rednote(product_name_2, tone_style_2)

print("\n--- 生成的文案 2 ---")
print(result_2)


🚀 启动小红书文案生成助手，产品：美白精华，风格：知性温柔

-- Iteration 1 --
Agent: 决定调用工具...
Agent Action: 调用工具 'query_product_database'，参数：{'product_name': '美白精华'}
[Tool Call] 模拟查询产品数据库：美白精华
Observation: 工具返回结果：美白精华：核心成分是烟酰胺和VC衍生物，主要功效是提亮肤色、淡化痘印、改善暗沉。质地轻薄易吸收，适合需要均匀肤色的人群。
-- Iteration 2 --
Agent: 决定调用工具...
Agent Action: 调用工具 'generate_emoji'，参数：{'context': '美白精华提亮肤色'}
[Tool Call] 模拟生成表情符号，上下文：美白精华提亮肤色
Observation: 工具返回结果：['💧']
-- Iteration 3 --
[模型生成结果] ```json
{
  "title": "✨一瓶拯救暗沉肌！我的美白精华空瓶记✨",
  "body": "姐妹们！终于找到了一瓶让我肤色亮到发光的美白精华！💖\n\n核心成分是烟酰胺+VC衍生物，双管齐下，提亮肤色效果真的绝了！💧\n\n🌟质地轻薄到爆，一抹就吸收，完全不会黏腻！\n🌟坚持用了两周，痘印淡了好多，整张脸都透亮了！\n🌟熬夜党的救星，暗沉肌的克星！\n\n真心推荐给所有需要均匀肤色的姐妹，入股不亏！🔥\n\n#美白精华 #提亮肤色 #烟酰胺 #VC精华 #空瓶记",
  "hashtags": ["#美白精华", "#提亮肤色", "#烟酰胺", "#VC精华", "#空瓶记"],
  "emojis": ["✨", "💖", "💧", "🌟", "🔥"]
}
```
Agent: 任务完成，成功解析最终JSON文案。

--- 生成的文案 2 ---
{
  "title": "✨一瓶拯救暗沉肌！我的美白精华空瓶记✨",
  "body": "姐妹们！终于找到了一瓶让我肤色亮到发光的美白精华！💖\n\n核心成分是烟酰胺+VC衍生物，双管齐下，提亮肤色效果真的绝了！💧\n\n🌟质地轻薄到爆，一抹就吸收，完全不会黏腻！\n🌟坚持用了两周，痘印淡了好多，整张脸都透亮了！\n🌟熬夜党的救星，暗沉肌的克星！\

### 格式化 小红书文案

**格式化函数 `format_rednote_for_markdown` 的功能：**

1. 解析 JSON 字符串。
2. 提取标题、正文、标签和表情符号。
3. 将它们组合成一个易读的 Markdown 格式的文本。


**工作方式：**

1. **解析 JSON**：使用 `json.loads()` 将输入的字符串转换为 Python 字典。如果解析失败，会返回一个错误信息。
2. **提取数据**：使用 `.get()` 方法从字典中安全地提取 `title`、`body` 和 `hashtags`。使用 `.get()` 的好处是，如果某个键不存在，它会返回一个默认值（例如 `None` 或空列表），而不是抛出 `KeyError`。
3. **构建 Markdown 标题**：将 `title` 格式化为 Markdown 的二级标题 (`## Title`)。
4. **处理正文**：直接使用 `body`。由于小红书正文中的换行很重要，我们保留它们。
5. **处理 Hashtags**：将 `hashtags` 列表中的每个标签用空格连接起来，形成一行。
6. **表情符号 (Emojis)**：在小红书的实际发布中，表情符号通常已经嵌入在标题和正文中了。这个函数没有单独列出它们，因为这通常不是最终发布格式的一部分。如果需要，可以取消注释相关代码来单独显示它们。
7. **返回结果**：返回拼接好的 Markdown 字符串，并使用 `.strip()` 去除可能存在于末尾的多余空白。

In [20]:
import json

def format_rednote_for_markdown(json_string: str) -> str:
    """
    将 JSON 格式的小红书文案转换为 Markdown 格式，以便于阅读和发布。

    Args:
        json_string (str): 包含小红书文案的 JSON 字符串。
                           预计格式为 {"title": "...", "body": "...", "hashtags": [...], "emojis": [...]}

    Returns:
        str: 格式化后的 Markdown 文本。
    """
    try:
        data = json.loads(json_string)
    except json.JSONDecodeError as e:
        return f"错误：无法解析 JSON 字符串 - {e}\n原始字符串：\n{json_string}"

    title = data.get("title", "无标题")
    body = data.get("body", "")
    hashtags = data.get("hashtags", [])
    # 表情符号通常已经融入标题和正文中，这里可以选择是否单独列出
    # emojis = data.get("emojis", []) 

    # 构建 Markdown 文本
    markdown_output = f"## {title}\n\n" # 标题使用二级标题
    
    # 正文，保留换行符
    markdown_output += f"{body}\n\n"
    
    # Hashtags
    if hashtags:
        hashtag_string = " ".join(hashtags) # 小红书标签通常是空格分隔
        markdown_output += f"{hashtag_string}\n"
        
    # 如果需要，可以单独列出表情符号，但通常它们已经包含在标题和正文中
    # if emojis:
    #     emoji_string = " ".join(emojis)
    #     markdown_output += f"\n使用的表情：{emoji_string}\n"
        
    return markdown_output.strip() # 去除末尾多余的空白

In [21]:
# --- 示例使用 ---
# 假设这是 generate_rednote 函数的输出
generated_json_output = """
{
  "title": "🔥程序员福音！Cursor：新一代智能代码编辑器，效率翻倍不是梦！",
  "body": "作为一名资深程序员，最近被一款神器彻底征服了——Cursor！它不仅支持多语言编程，还内置了AI智能补全功能，简直是敲代码的加速器！🚀\n\n🌟 核心功能亮点：\n1. **AI智能补全**：根据上下文自动生成代码，告别重复劳动。\n2. **多语言支持**：Python、Java、C++...统统不在话下。\n3. **实时协作**：团队开发更高效，远程办公无压力。\n4. **轻量级设计**：启动快，运行稳，再也不用担心卡顿。\n\n💡 使用体验：\n上手第一天就爱上了！AI补全的准确率超高，写代码的时间直接缩短了一半。团队协作功能也超赞，再也不用频繁切换工具了！\n\n如果你也是程序员，或者对高效工具有追求，Cursor绝对是你的不二之选！快来试试吧～",
  "hashtags": [
    "#程序员必备",
    "#高效工具",
    "#AI编程",
    "#代码编辑器",
    "#科技神器"
  ],
  "emojis": [
    "🚀",
    "🌟",
    "💡",
    "🔥",
    "👨‍💻"
  ]
}
"""

# 调用格式化函数
markdown_note = format_rednote_for_markdown(generated_json_output)

# 打印结果
print("--- 格式化后的小红书文案 (Markdown) ---")
print(markdown_note)

# --- 另一个例子，假设JSON解析失败 ---
invalid_json_output = "{'title': 'Test', 'body': 'This is not valid json'}" # 使用单引号，非法
markdown_error_note = format_rednote_for_markdown(invalid_json_output)
print("\n--- 格式化错误示例 ---")
print(markdown_error_note)


--- 格式化后的小红书文案 (Markdown) ---
错误：无法解析 JSON 字符串 - Invalid control character at: line 4 column 79 (char 129)
原始字符串：

{
  "title": "🔥程序员福音！Cursor：新一代智能代码编辑器，效率翻倍不是梦！",
  "body": "作为一名资深程序员，最近被一款神器彻底征服了——Cursor！它不仅支持多语言编程，还内置了AI智能补全功能，简直是敲代码的加速器！🚀

🌟 核心功能亮点：
1. **AI智能补全**：根据上下文自动生成代码，告别重复劳动。
2. **多语言支持**：Python、Java、C++...统统不在话下。
3. **实时协作**：团队开发更高效，远程办公无压力。
4. **轻量级设计**：启动快，运行稳，再也不用担心卡顿。

💡 使用体验：
上手第一天就爱上了！AI补全的准确率超高，写代码的时间直接缩短了一半。团队协作功能也超赞，再也不用频繁切换工具了！

如果你也是程序员，或者对高效工具有追求，Cursor绝对是你的不二之选！快来试试吧～",
  "hashtags": [
    "#程序员必备",
    "#高效工具",
    "#AI编程",
    "#代码编辑器",
    "#科技神器"
  ],
  "emojis": [
    "🚀",
    "🌟",
    "💡",
    "🔥",
    "👨‍💻"
  ]
}


--- 格式化错误示例 ---
错误：无法解析 JSON 字符串 - Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
原始字符串：
{'title': 'Test', 'body': 'This is not valid json'}


In [22]:
# 调用格式化函数
markdown_note = format_rednote_for_markdown(result_1)

# 打印结果
print("--- 格式化后的小红书文案 (Markdown) ---")
print(markdown_note)

--- 格式化后的小红书文案 (Markdown) ---
## 🔥程序员福音！Cursor：新一代智能代码编辑器，效率翻倍不是梦！

作为一名资深程序员，最近被一款神器彻底征服了——Cursor！它不仅支持多语言编程，还内置了AI智能补全功能，简直是敲代码的加速器！🚀

🌟 核心功能亮点：
1. **AI智能补全**：根据上下文自动生成代码，告别重复劳动。
2. **多语言支持**：Python、Java、C++...统统不在话下。
3. **实时协作**：团队开发更高效，远程办公无压力。
4. **轻量级设计**：启动快，运行稳，再也不用担心卡顿。

💡 使用体验：
上手第一天就爱上了！AI补全的准确率超高，写代码的时间直接缩短了一半。团队协作功能也超赞，再也不用频繁切换工具了！

如果你也是程序员，或者对高效工具有追求，Cursor绝对是你的不二之选！快来试试吧～

#程序员必备 #高效工具 #AI编程 #代码编辑器 #科技神器


## 6. 评估与优化

文案生成并非一蹴而就，需要持续的评估和优化。本节讨论一些评估方法和优化策略。

#### 评估文案质量：
*   **客观量化评估 (数据)：**
    *   **点赞/收藏/评论/分享：** 基础互动
    *   **曝光/阅读/点击/涨粉：：** 流量与曝光
    *   **停留时长/截图率：** 用户行为。
    *   **商品页浏览/加购/ROI/成交转化：** 商业价值
    *   **爆文率/同类横向对比：** 竞争对比
*   **主观内部评估 (人工)：**
    *   **相关性：** 是否符合产品特点和主题。
    *   **吸引力：** 标题是否抓人，内容是否流畅。
    *   **合规性：** 是否有敏感词、违规宣传。
    *   **风格匹配：** 是否符合小红书调性和指定语气。
    *   **用户画像：** 目标人群年龄、地域、兴趣标签。



#### 优化迭代方法：
*   **Prompt 调整：** 根据评估结果，精修 System Prompt、User Prompt，增加或修改 Few-shot 示例。
*   **工具扩充：** 引入新的工具（如敏感词检测工具、竞品分析工具）。
*   **RAG (检索增强生成)：** 结合更精准的内部知识库，减少幻觉。


## 7. 总结与展望

通过本次实战，我们成功构建了一个基于 DeepSeek Agent 的小红书爆款文案生成助手。我们学习了如何拆解需求、设计 Prompt、定义工具，并实现 Agent 的核心工作流。

Agent 在内容营销领域的潜力巨大，未来可以进一步拓展到：

*   **超个性化内容：** 根据用户数据，生成一对一的定制文案。
*   **多模态内容创作：** 结合图片、视频生成，实现图文音视频一体化。
*   **智能营销决策：** Agent 不仅生成内容，还能分析效果并给出投放建议。
*   **跨平台适配：** 快速生成适应不同社交媒体平台风格的文案。

同时，我们也需关注挑战，如确保内容真实性、处理高度主观情感、与现有工作流的无缝集成等。Agent 技术仍在快速发展，期待未来能带来更多惊喜！